# Financial Data with AKShare

AKShare is a Python-based financial data interface library that can be used to collect, clean, and analyze Chinese market data for academic or teaching purposes.

**Note:** This notebook is China-market specific. Because online financial-data APIs can change over time, some AKShare endpoints may occasionally change or become temporarily unavailable.

In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT


In [ ]:
MODULE_OUTPUT_DIR = OUTPUT_DIR / "06_financial_data"
MODULE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODULE_OUTPUT_DIR

**Optional setup**

In [ ]:
%pip install -q akshare ipywidgets seaborn scipy

In [ ]:
import akshare as ak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

TRADING_DAYS = 242
RISK_FREE_RATE = 0.015

def to_ak_tx_symbol(symbol: str) -> str:
    if symbol.startswith(("5", "6", "9")):
        return f"sh{symbol}"
    return f"sz{symbol}"

def fetch_stock_history(symbol: str, start_date: str, end_date: str, adjust: str = "hfq") -> pd.DataFrame:
    data = ak.stock_zh_a_hist_tx(
        symbol=to_ak_tx_symbol(symbol),
        start_date=start_date,
        end_date=end_date,
        adjust=adjust,
    ).rename(
        columns={
            "date": "date",
            "open": "open",
            "close": "close",
            "high": "high",
            "low": "low",
            "amount": "amount",
        }
    )
    data["date"] = pd.to_datetime(data["date"])
    return data

# Part 1. Obtain stock data and compute basic statistics

# Q1. How can we check the stock codes and names of listed companies on the A-share market?

For `stock_info_sh_name_code`, the `symbol` parameter uses Chinese board names such as:
- `主板A股` = Main Board A-shares
- `主板B股` = Main Board B-shares
- `科创板` = STAR Market

In [ ]:
board_symbol = "主板A股"
stock_info = ak.stock_info_sh_name_code(symbol=board_symbol)
stock_info.head()

In [ ]:
# Reproducible recent snapshot for a small A-share watchlist.
snapshot_symbols = ["600000", "600004", "600006", "600007"]
snapshot_rows = []

for symbol in snapshot_symbols:
    latest_row = fetch_stock_history(symbol, start_date="20241201", end_date="20250101").iloc[-1]
    snapshot_rows.append(
        {
            "stock_code": symbol,
            "date": latest_row["date"],
            "open": latest_row["open"],
            "close": latest_row["close"],
            "high": latest_row["high"],
            "low": latest_row["low"],
            "amount": latest_row["amount"],
        }
    )

stock_today = pd.DataFrame(snapshot_rows).merge(
    stock_info[["证券代码", "证券简称"]].rename(columns={"证券代码": "stock_code", "证券简称": "stock_name"}),
    on="stock_code",
    how="left",
)

stock_today = stock_today[["stock_code", "stock_name", "date", "open", "close", "high", "low", "amount"]]
stock_today

# Q2. How can we obtain stock data?

## Q2.1. Single-stock example: Shanghai Pudong Development Bank (`600000`)

In [ ]:
stock_pufa = fetch_stock_history(
    symbol="600000",
    start_date="20240101",
    end_date="20250101",
    adjust="hfq",
)
stock_pufa["daily_return"] = stock_pufa["close"].pct_change()

stock_pufa.head()

## Q2.2. Multi-stock example

In [ ]:
symbols = ["600000", "600004", "600006", "600007"]

stock_name_map = {
    "600000": "Shanghai Pudong Development Bank",
    "600004": "Baiyun Airport",
    "600006": "Dongfeng Motor",
    "600007": "China World Trade Center",
}

dfs = []
for symbol in symbols:
    data = fetch_stock_history(
        symbol=symbol,
        start_date="20240101",
        end_date="20241231",
        adjust="hfq",
    )

    data["stock_code"] = symbol
    data["stock_name"] = stock_name_map[symbol]
    data["daily_return"] = data["close"].pct_change()
    dfs.append(data)

stock_df = pd.concat(dfs, ignore_index=True)
stock_df.head()

# Q3. How can we calculate annualized return, annualized volatility, and the Sharpe ratio?

Assume 242 trading days per year.

- Annualized return = mean daily return × 242
- Annualized volatility = standard deviation of daily return × √242
- Sharpe ratio = (annualized return − risk-free rate) / annualized volatility

This notebook uses an annualized risk-free rate of 0.015 (1.5%).

In [ ]:
stock_pufa[["date", "close", "daily_return"]].head()

In [ ]:
pufa_return = stock_pufa["daily_return"].mean() * TRADING_DAYS
pufa_return

In [ ]:
pufa_vol = stock_pufa["daily_return"].std() * np.sqrt(TRADING_DAYS)
pufa_vol

In [ ]:
pufa_sharpe = (pufa_return - RISK_FREE_RATE) / pufa_vol
pufa_sharpe

# Q4. Grouped descriptive statistics

In [ ]:
summary_table = stock_df.groupby("stock_name")[["daily_return", "close"]].describe().round(4).T
summary_table

# Part 2. Visualize stock data

# Q5. How can we create basic visualizations for stock data?

## Q5.1. Opening and closing prices for Shanghai Pudong Development Bank

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(stock_pufa["date"], stock_pufa["open"], linestyle="-.", linewidth=1, marker=".", color="#0f0f0f80")
plt.plot(stock_pufa["date"], stock_pufa["close"], color="yellow")
plt.xlabel("Date")
plt.ylabel("Price")
plt.title("Shanghai Pudong Development Bank: Open vs. Close")
plt.legend(["Opening price", "Closing price"], loc="best")
plt.savefig(MODULE_OUTPUT_DIR / "open_close_lineplot.png", bbox_inches="tight")

In [ ]:
%pip install -q seaborn

## Q5.2. Multi-stock line plot

In [ ]:
import seaborn as sns

plt.figure(figsize=(12, 5))
sns.lineplot(data=stock_df, x="date", y="daily_return", hue="stock_name")
plt.title("Daily Return Series")
plt.xlabel("Date")
plt.ylabel("Daily return")
plt.tight_layout()
plt.savefig(MODULE_OUTPUT_DIR / "multi_stock_daily_return_lineplot.png", bbox_inches="tight")
plt.show()

## Q5.3. Histograms for multiple stocks

In [ ]:
g = sns.FacetGrid(stock_df.dropna(subset=["daily_return"]), col="stock_name", col_wrap=2, sharex=False, sharey=False)
g.map(sns.histplot, "daily_return", color="pink", bins=20)
g.set_titles("{col_name}")
g.fig.tight_layout()

## Q5.4. Cumulative return plot

In [ ]:
stock_df = stock_df.sort_values(["stock_name", "date"]).copy()
stock_df["cumulative_return"] = (
    stock_df.groupby("stock_name")["daily_return"]
    .transform(lambda s: (1 + s.fillna(0)).cumprod() - 1)
)

plt.figure(figsize=(12, 5))
sns.lineplot(data=stock_df, x="date", y="cumulative_return", hue="stock_name")
plt.title("Compounded Cumulative Return")
plt.xlabel("Date")
plt.ylabel("Cumulative return")
plt.tight_layout()
plt.savefig(MODULE_OUTPUT_DIR / "cumulative_return_lineplot.png", bbox_inches="tight")
plt.show()

# Part 3. Markowitz mean-variance portfolio model

This section uses Monte Carlo simulation and constrained optimization to explore random portfolios, the maximum-Sharpe portfolio, the minimum-variance portfolio, and the efficient frontier.

In [ ]:
from scipy.optimize import minimize
import scipy.optimize as sco

returns = (
    stock_df[["stock_code", "daily_return", "date"]]
    .pivot_table(values="daily_return", index="date", columns="stock_code")
    .dropna(how="all")
    .dropna()
)
returns.head()

# Q6. How can we generate many random portfolios with Monte Carlo simulation?

In [ ]:
noa = len(symbols)

portfolio_returns = []
portfolio_volatility = []
portfolio_sharpe = []
portfolio_weights = []

for _ in range(5000):
    weights = np.random.random(noa)
    weights /= np.sum(weights)

    port_return = np.sum(returns.mean() * TRADING_DAYS * weights)
    port_volatility = np.sqrt(np.dot(weights.T, np.dot(returns.cov() * TRADING_DAYS, weights)))
    sharpe_ratio = (port_return - RISK_FREE_RATE) / port_volatility

    portfolio_returns.append(port_return)
    portfolio_volatility.append(port_volatility)
    portfolio_sharpe.append(sharpe_ratio)
    portfolio_weights.append(weights)

portfolio_returns = np.array(portfolio_returns)
portfolio_volatility = np.array(portfolio_volatility)
portfolio_sharpe = np.array(portfolio_sharpe)

In [ ]:
plt.figure(figsize=(8, 4))
scatter = plt.scatter(portfolio_volatility, portfolio_returns, c=portfolio_sharpe, marker="o", alpha=0.7)
plt.xlabel("Expected volatility")
plt.ylabel("Expected return")
plt.grid(True)
plt.colorbar(scatter, label="Sharpe ratio")
plt.tight_layout()
plt.savefig(MODULE_OUTPUT_DIR / "monte_carlo_portfolios.png", bbox_inches="tight")
plt.show()

# Q7. How can we find the portfolio with the largest Sharpe ratio?

In [ ]:
max_sharpe_idx = np.argmax(portfolio_sharpe)
max_sharpe_return = portfolio_returns[max_sharpe_idx]
max_sharpe_volatility = portfolio_volatility[max_sharpe_idx]
max_sharpe_ratio = portfolio_sharpe[max_sharpe_idx]

print(f"Largest Sharpe ratio: {max_sharpe_ratio:.4f}")
print(f"Expected return: {max_sharpe_return:.4f}")
print(f"Expected volatility: {max_sharpe_volatility:.4f}")

In [ ]:
def statistics(weights):
    weights = np.array(weights)
    port_return = np.sum(returns.mean() * TRADING_DAYS * weights)
    port_volatility = np.sqrt(np.dot(weights.T, np.dot(returns.cov() * TRADING_DAYS, weights)))
    sharpe = (port_return - RISK_FREE_RATE) / port_volatility
    return np.array([port_return, port_volatility, sharpe])

def negative_sharpe(weights):
    return -statistics(weights)[2]

bounds = ((0, 1),) * noa
constraints = {"type": "eq", "fun": lambda x: np.sum(x) - 1}

opt_sharpe = sco.minimize(
    negative_sharpe,
    noa * [1.0 / noa],
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
)

opt_sharpe

# Q8. How can we find the portfolio that minimizes variance?

In [ ]:
def portfolio_vol(weights):
    return statistics(weights)[1]

opt_min_vol = sco.minimize(
    portfolio_vol,
    noa * [1.0 / noa],
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
)

print(f"Optimal portfolio weights: {opt_min_vol.x}")
print(f"Minimum volatility: {opt_min_vol.fun:.4f}")

# Q9. How can we draw the efficient frontier?

In [ ]:
target_returns = np.linspace(
    (returns.mean() * TRADING_DAYS).min(),
    (returns.mean() * TRADING_DAYS).max(),
    100,
)

target_volatility = []

for target in target_returns:
    target_constraints = (
        {"type": "eq", "fun": lambda x, target=target: statistics(x)[0] - target},
        {"type": "eq", "fun": lambda x: np.sum(x) - 1},
    )

    result = sco.minimize(
        portfolio_vol,
        noa * [1.0 / noa],
        method="SLSQP",
        bounds=bounds,
        constraints=target_constraints,
    )

    target_volatility.append(result["fun"])

target_volatility = np.array(target_volatility)

In [ ]:
plt.figure(figsize=(8, 4))
scatter = plt.scatter(portfolio_volatility, portfolio_returns, c=portfolio_sharpe, marker="o", alpha=0.7)
plt.scatter(target_volatility, target_returns, c=target_returns / target_volatility, cmap="autumn", marker="x", s=8)
plt.plot(statistics(opt_sharpe["x"])[1], statistics(opt_sharpe["x"])[0], "r*", markersize=15)
plt.plot(statistics(opt_min_vol["x"])[1], statistics(opt_min_vol["x"])[0], "y*", markersize=15)
plt.xlabel("Expected volatility")
plt.ylabel("Expected return")
plt.title("Efficient Frontier")
plt.grid(True)
plt.colorbar(scatter, label="Sharpe ratio")
plt.tight_layout()
plt.savefig(MODULE_OUTPUT_DIR / "efficient_frontier.png", bbox_inches="tight")
plt.show()